## **Assignment 5: Fine-Tuning BERT for POS Tagging & Chunking**

### Installing Required Libraries

In [ ]:
!pip install transformers seqeval evaluate conllu accelerate datasets -q

### Importing Libraries

In [ ]:
import numpy as np
import torch  # PyTorch for model training

from transformers import (
    AutoTokenizer,  # Loads pretrained tokenizer
    AutoModelForTokenClassification,  # Model for token-level tasks
    TrainingArguments,  # Training configuration
    Trainer,  # Trainer API to handle training loop
    DataCollatorForTokenClassification  # Handles batching & padding for token tasks
)

from evaluate import load  # For evaluation metrics
from conllu import parse  # To parse CoNLL-U formatted datasets

### Downloading the Universal Dependencies dataset

In [ ]:
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-train.conllu
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-dev.conllu
!wget https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/en_ewt-ud-test.conllu

### Loading Function for the Dataset

In [ ]:
def load_ud_data(file_path):
    tokens_list = []  # List to store tokens of all sentences
    pos_tags_list = []  # List to store POS tags of all sentences

    # Open and read the file, parse it using conllu
    with open(file_path, "r", encoding="utf-8") as f:
        sentences = parse(f.read())

    for sentence in sentences:
        tokens = []  # Tokens for this sentence
        pos_tags = []  # POS tags for this sentence

        # Extract tokens and POS tags
        for token in sentence:
            tokens.append(token["form"])  # Word/token text
            pos_tags.append(token["upos"])  # Universal POS tag

        # Add sentence tokens and tags to main lists
        tokens_list.append(tokens)
        pos_tags_list.append(pos_tags)

    return tokens_list, pos_tags_list  # Return all sentences and their POS tags

### Loading and Preparing UD Dataset

In [ ]:
# Load training and validation data
train_sentences, train_labels = load_ud_data("/content/en_ewt-ud-train.conllu.1")
val_sentences, val_labels = load_ud_data("/content/en_ewt-ud-dev.conllu.1")

# Use only a subset for faster experimentation
train_sentences = train_sentences[:4000]
train_labels = train_labels[:4000]

val_sentences = val_sentences[:500]
val_labels = val_labels[:500]

# Preview first sentence and its POS tags
print(train_sentences[0])
print(train_labels[0])

### Preparing Label Mappings for POS Tags

In [ ]:
# Extract unique POS tags from training labels
unique_tags = sorted(list(set(tag for sent in train_labels for tag in sent)))

# Create mapping from label to ID and vice versa
tag2id = {tag: idx for idx, tag in enumerate(unique_tags)}
id2tag = {idx: tag for tag, idx in tag2id.items()}

# Total number of labels
num_tags = len(unique_tags)

# Preview all unique POS tags
print(unique_tags)

### Loading BERT Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

### Tokenization & Label Alignment

In [ ]:
def tokenize_and_align(sentences, tags):
    """
    Tokenize sentences and align POS tags with tokens.
    Subword tokens get -100 label to ignore in loss computation.
    """

    # Tokenize input sentences (list of words)
    tokenized_output = tokenizer(
        sentences,
        truncation=True,
        is_split_into_words=True
    )

    # Map each token back to its word index
    word_indices = tokenized_output.word_ids()

    aligned_labels = []
    prev_word_idx = None

    # Align labels with tokens
    for word_idx in word_indices:
        if word_idx is None:
            # Special tokens get -100
            aligned_labels.append(-100)
        elif word_idx != prev_word_idx:
            # Assign label to first subword of the word
            aligned_labels.append(tag2id[tags[word_idx]])
        else:
            # Remaining subwords get -100
            aligned_labels.append(-100)

        prev_word_idx = word_idx

    # Add aligned labels to tokenized output
    tokenized_output["labels"] = aligned_labels

    return tokenized_output

### Data Preprocessing

In [ ]:
train_data = [tokenize_and_align(sent, tags) for sent, tags in zip(train_sentences, train_labels)]
val_data = [tokenize_and_align(sent, tags) for sent, tags in zip(val_sentences, val_labels)]

### Creating PyTorch Dataset for Token Classification

In [ ]:
class TokenDataset(torch.utils.data.Dataset):

    def __init__(self, encodings):
        self.encodings = encodings  # Store tokenized inputs and labels

    def __getitem__(self, idx):
        # Convert each item (input_ids, attention_mask, labels) to tensors
        item = {key: torch.tensor(val) for key, val in self.encodings[idx].items()}
        return item

    def __len__(self):
        return len(self.encodings)  # Number of samples in dataset

# Create datasets for training and validation
train_dataset = TokenDataset(train_data)
val_dataset = TokenDataset(val_data)

### Loading Pretrained BERT for Token Classification

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",  # Pretrained BERT model
    num_labels=num_tags,  # Number of POS tags
    id2label=id2tag,      # Map from label IDs to tag names
    label2id=tag2id       # Map from tag names to label IDs
)

### Preparing the Data Collator

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

### Evaluation Metrics Function


In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Take argmax over prediction logits to get predicted class IDs
    predictions = np.argmax(predictions, axis=2)

    # Align predictions with true labels, ignore subword tokens (-100)
    true_preds = [
        [id2tag[pred_id] for pred_id, label_id in zip(pred_seq, label_seq) if label_id != -100]
        for pred_seq, label_seq in zip(predictions, labels)
    ]

    true_labels = [
        [id2tag[label_id] for pred_id, label_id in zip(pred_seq, label_seq) if label_id != -100]
        for pred_seq, label_seq in zip(predictions, labels)
    ]

    # Compute metrics using evaluate library
    results = metric.compute(
        predictions=true_preds,
        references=true_labels
    )

    # Return key metrics
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

### Defining Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./pos_model",       # Directory to save model checkpoints
    learning_rate=2e-5,             # Learning rate for optimizer
    per_device_train_batch_size=16, # Batch size for training
    per_device_eval_batch_size=16,  # Batch size for evaluation
    num_train_epochs=3,             # Number of training epochs
    weight_decay=0.01,              # L2 weight decay
    logging_steps=100               # Log metrics every 100 steps
)

### Initializing Hugging Face Trainer

In [ ]:
trainer = Trainer(
    model=model,                     # The token classification model
    args=training_args,              # Training configuration
    train_dataset=train_dataset,     # Training dataset
    eval_dataset=val_dataset,        # Validation dataset
    data_collator=data_collator,     # Handles batching & padding
    compute_metrics=compute_metrics  # Function to compute evaluation metrics
)

### Model Training

In [ ]:
trainer.train()

### Evaluation Metrics

In [ ]:
metric = load("seqeval")
trainer.evaluate()

Inference Example - 1

In [ ]:
# ===== POS Tagging Example on a Single Sentence =====
sentence = "John works at Google in California"

# Select device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Tokenize the input sentence
inputs = tokenizer(sentence, return_tensors="pt")
inputs = {key: val.to(device) for key, val in inputs.items()}

# Make predictions without computing gradients
with torch.no_grad():
    outputs = model(**inputs)

# Take argmax over logits to get predicted label IDs
pred_ids = torch.argmax(outputs.logits, dim=2)

# Convert token IDs back to tokens
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu())

# Map predicted IDs to POS tags
pred_tags = [id2tag[p.item()] for p in pred_ids[0]]

# Display tokens with their predicted POS tags
for token, tag in zip(tokens, pred_tags):
    print(token, tag)

Inference Example - 2

In [ ]:
example_sentence = "Alice enjoys reading books at the library"

# Set device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Tokenize the example sentence
inputs = tokenizer(example_sentence, return_tensors="pt")
inputs = {key: val.to(device) for key, val in inputs.items()}

# Forward pass without gradient calculation
with torch.no_grad():
    outputs = model(**inputs)

# Get predicted label IDs
pred_ids = torch.argmax(outputs.logits, dim=2)

# Convert input IDs back to tokens
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu())

# Map predicted IDs to tag names
pred_tags = [id2tag[p.item()] for p in pred_ids[0]]

# Print tokens with their predicted POS tags
for token, tag in zip(tokens, pred_tags):
    print(token, tag)

### Comparision Table

| Feature        | POS Tagging                     | Chunking                     |
| -------------- | ------------------------------- | ---------------------------- |
| **Level**      | Word-level                      | Phrase-level                 |
| **Output**     | Grammar tags (e.g., NOUN, VERB) | Phrase groups (e.g., NP, VP) |
| **Complexity** | Easy                            | Medium                       |


### **Observations:**
- BERT-based token classification gives high accuracy even on small subsets of data.
- Subword tokenization requires careful alignment of labels with tokens.
- Most common POS tags (NOUN, VERB, ADJ) are predicted more accurately than rarer ones.
- Pre-trained models can generalize to unseen words due to WordPiece embeddings.

### **Conclusion:**

Fine-tuning pre-trained transformer models like BERT for POS tagging proves to be highly effective, even with a relatively small subset of data. Careful alignment of labels with tokenized subwords is essential to ensure accurate model training and evaluation. The model demonstrates strong performance on common POS tags, highlighting its ability to generalize to unseen words, while challenges with rarer tags emphasize the importance of balanced datasets. Overall, POS tagging not only provides fundamental insights into sentence structure but also serves as a crucial building block for higher-level NLP tasks such as chunking, parsing, and information extraction, making it a valuable step in natural language understanding pipelines.